In [0]:
%sql
-- Paso 1 (estructura), mirada global: cantidad de columnas de cada tabla bronze
-- y sus primeras 6, para ubicar dónde está la columna de país, de año, etc.
SELECT table_name,
       COUNT(*) AS columnas,
       CONCAT_WS(', ', SLICE(ARRAY_SORT(COLLECT_LIST(STRUCT(ordinal_position, column_name))).column_name, 1, 6)) AS primeras_6
FROM happiness.information_schema.columns
WHERE table_schema = 'bronze'
GROUP BY table_name
ORDER BY table_name;

In [0]:
%sql
SELECT COUNT(*) AS filas FROM happiness.bronze.wgi;

DESCRIBE TABLE happiness.bronze.wgi;

SELECT * FROM happiness.bronze.wgi LIMIT 5;

WGI, paso 1. Son 22.540 filas, en formato largo: una por economía, dimensión de gobernanza y año. Economy (code) parece ser ISO3 (AFG, AGO). Los valores faltantes vienen como el texto .., no como NULL, así que en Silver hay que convertir .. a NULL antes del CAST. Hay unas 40 columnas de fuentes individuales (ADB mean, AFR mean...), casi todas vacías. La columna Region sale null para Andorra, y no sé aún si es NULL real o texto.

In [0]:
%sql
-- WGI · Paso 2 (nulos): ¿cómo se marcan los faltantes? Puede ser NULL real, el texto 'null' o '..'.
-- Se mira en las claves, en la región y en el estimado principal (la variable que usará el proyecto).
SELECT COUNT(*)                                                     AS filas,
       COUNT_IF(`Economy (code)` IS NULL)                           AS sin_codigo,
       COUNT_IF(`Governance dimension` IS NULL)                     AS sin_dimension,
       COUNT_IF(`Year` IS NULL)                                     AS sin_anio,
       COUNT_IF(`Region` IS NULL)                                   AS region_null_real,
       COUNT_IF(`Region` = 'null')                                  AS region_texto_null,
       COUNT_IF(`Governance estimate (approx. -2.5 to +2.5)` IS NULL)  AS estimado_null_real,
       COUNT_IF(`Governance estimate (approx. -2.5 to +2.5)` = '..')   AS estimado_puntos,
       COUNT_IF(`Governance estimate (approx. -2.5 to +2.5)` = 'null') AS estimado_texto_null,
       COUNT_IF(_rescued_data IS NOT NULL)                          AS filas_rescatadas
FROM happiness.bronze.wgi;

In [0]:
%sql
-- WGI · Paso 3 (cardinalidad): cuántas economías, dimensiones y años distintos hay.
-- Esperado: ~214-216 economías (S2), años 1996-2025. Si hay más nombres que códigos (o al revés),
-- una misma economía está escrita de dos formas.
SELECT COUNT(DISTINCT `Economy (code)`)         AS codigos_economia,
       COUNT(DISTINCT `Economy (name)`)         AS nombres_economia,
       COUNT(DISTINCT `Governance dimension`)   AS dimensiones,
       COUNT(DISTINCT `Year`)                   AS anios,
       MIN(`Year`)                              AS primer_anio,
       MAX(`Year`)                              AS ultimo_anio
FROM happiness.bronze.wgi;

In [0]:
%sql
-- WGI · Paso 3 (cardinalidad): filas por dimensión. Chequea que las 4 dimensiones elegidas
-- (va, pv, ge, rl) estén y cuántas otras trae el archivo (cc, rq...).
SELECT `Governance dimension` AS dimension, COUNT(*) AS filas, COUNT(DISTINCT `Economy (code)`) AS economias
FROM happiness.bronze.wgi
GROUP BY `Governance dimension`
ORDER BY dimension;

In [0]:
%sql
-- WGI · Paso 4 (rangos): estimado en la escala -2.5 a +2.5 y puntaje 0-100.
-- TRY_CAST devuelve NULL si el texto no es número; 'no_numericos' no cuenta los '..' (faltante esperado).
SELECT `Governance dimension` AS dimension,
       MIN(TRY_CAST(`Governance estimate (approx. -2.5 to +2.5)` AS DOUBLE)) AS estimado_min,
       MAX(TRY_CAST(`Governance estimate (approx. -2.5 to +2.5)` AS DOUBLE)) AS estimado_max,
       MIN(TRY_CAST(`Governance score (0-100)` AS DOUBLE))                  AS puntaje_min,
       MAX(TRY_CAST(`Governance score (0-100)` AS DOUBLE))                  AS puntaje_max,
       COUNT_IF(`Governance estimate (approx. -2.5 to +2.5)` NOT IN ('..', 'null')
                AND TRY_CAST(`Governance estimate (approx. -2.5 to +2.5)` AS DOUBLE) IS NULL) AS no_numericos,
       COUNT_IF(`Governance estimate (approx. -2.5 to +2.5)` = '..') AS faltantes
FROM happiness.bronze.wgi
GROUP BY `Governance dimension`
ORDER BY dimension;

In [0]:
%sql
-- WGI · Paso 5 (duplicados): la clave natural es (economía, dimensión, año).
-- ROW_NUMBER > 1 marca una clave repetida. Esperado: 0.
WITH numerado AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY `Economy (code)`, `Governance dimension`, `Year` ORDER BY _ingest_ts) AS rn
  FROM happiness.bronze.wgi
)
SELECT COUNT(*) AS filas, COUNT_IF(rn > 1) AS claves_duplicadas FROM numerado;

In [0]:
%sql
-- WGI · Paso 6 (temporal): cuántas economías tienen dato por año, para la dimensión 'va'.
-- Sirve para ver si faltan años (WGI históricamente no publicó 1997, 1999 y 2001) y hasta cuándo llega.
SELECT `Year` AS anio, COUNT(DISTINCT `Economy (code)`) AS economias
FROM happiness.bronze.wgi
WHERE `Governance dimension` = 'va'
GROUP BY `Year`
ORDER BY anio;

In [0]:
%sql
-- WGI · Hallazgo para dim_pais_base: economías sin región. Suelen ser territorios o agregados
-- que no entran como país (o países chicos que WGI no clasifica). Se anotan para revisar a mano.
SELECT DISTINCT `Economy (code)` AS codigo, `Economy (name)` AS nombre, `Region` AS region, `Income classification` AS ingreso
FROM happiness.bronze.wgi
WHERE `Region` IS NULL OR `Region` = 'null'
ORDER BY codigo;

## Conclusiones WGI (bronze.wgi, 22.540 filas)
- Formato largo: una fila por economía × dimensión × año. 216 economías (código = nombre, 1 a 1), 4 dimensiones (va, pv, ge, rl). `ge` trae 214 economías, las otras 216.
- Sin duplicados en (economía, dimensión, año). Sin nulos en claves ni `_rescued_data`.
- Estimado y puntaje: sin faltantes. Estimado entre -3,0 y 2,3; puntaje entre 1,5 y 99,6. Rangos coherentes.
- Los `..` solo están en las columnas por fuente (`ADB mean`...), que no se usan: se descartan en Silver.
- Años: 27 (1996, 1998, 2000 y 2002-2025). No hay 1997, 1999 ni 2001; no afecta el panel desde 2011. Cobertura 201-214 economías por año (208 en 2018 y 2025).
- Para dim_pais_base: 9 economías sin región (714 filas): Anguilla (AIA), Andorra (AND), Antillas Neerlandesas (ANT, ya disuelta), Islas Cook (COK), Guayana Francesa (GUF), Jersey (JEY), Martinica (MTQ), Niue (NIU) y Reunión (REU). Son territorios, no agregados. Es probable que WHR no los traiga. "Réunion" llegó con la tilde rota (codificación): cruzar por código, no por nombre.

Celda extra, solo para dim_pais_base: compara los códigos de WGI con los de Banco Mundial. WGI históricamente usa códigos viejos (por ejemplo ROM, ZAR, TMP, WBG). Si hay diferencias, hay que decidir en qué código nos quedamos.

In [0]:
%sql
-- WGI vs Banco Mundial · Códigos de economía que no coinciden entre las dos fuentes.
-- Si una misma economía usa códigos distintos, un JOIN por código la perdería sin avisar.
-- 'solo_wgi' = el código está en WGI y no en Banco Mundial; 'solo_bm' = al revés.
WITH wgi AS (
  SELECT DISTINCT `Economy (code)` AS codigo, `Economy (name)` AS nombre FROM happiness.bronze.wgi
),
bm AS (
  SELECT DISTINCT `Country Code` AS codigo, `Country Name` AS nombre FROM happiness.bronze.banco_mundial
)
SELECT 'solo_wgi' AS donde, w.codigo, w.nombre
FROM wgi w LEFT ANTI JOIN bm ON w.codigo = bm.codigo
UNION ALL
SELECT 'solo_bm', b.codigo, b.nombre
FROM bm b LEFT ANTI JOIN wgi w ON b.codigo = w.codigo
ORDER BY donde, codigo;

El cruce WGI vs Banco Mundial salió bien. Hay 207 códigos que coinciden. No aparecen códigos viejos como ROM, ZAR, TMP ni WBG, y Kosovo (XKX) y Cisjordania/Gaza (PSE) también coinciden. Eso  sirve para dim_pais_base.

Lo que quedó fuera del cruce:

- Solo en Banco Mundial (59): 49 son agregados (WLD, HIC, EUU, SSF, INX...). Los otros 10 son territorios con ISO3 válido: Islas del Canal, Curazao, Feroe, Gibraltar, Isla de Man, San Martín (parte francesa), Marianas del Norte, Sint Maarten, Turcas y Caicos, e Islas Vírgenes Británicas.
- Solo en WGI (9): los 8 territorios que ya conocíamos (Anguilla, Antillas Neerlandesas, Islas Cook, Guayana Francesa, Jersey, Martinica, Niue y Reunión), más Taiwán (TWN). Banco Mundial no lo incluye. WGI lo trae como "Taiwan, China", y WHR probablemente como "Taiwan Province of China". Habrá que cruzarlo por nombre y asignarle TWN a mano. ANT es un código retirado desde 2010 y no va a la tabla final.
- Reunión llega con la tilde rota en WGI, así que conviene cruzar siempre por código.